# ChartScope — Fine-tune PubMedBERT for Clinical NER

This notebook fine-tunes **PubMedBERT** on the public [**NCBI-Disease**](https://huggingface.co/datasets/ncbi/ncbi_disease) corpus and benchmarks it against the ChartScope baseline (`d4data/biomedical-ner-all`).

**Expected outcome:** entity-level F1 in the **high 0.80s** on the test split, beating the general baseline (~0.55–0.70).

**Runtime:** ~10 minutes on a free Colab T4 GPU (3 epochs).

> Data governance: NCBI-Disease is public and requires no credentialing. MIMIC/n2c2/i2b2 remain offline-only per ChartScope policy.

## 1. Setup GPU + clone repo

Enable **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi || echo 'No GPU detected — training will be slow on CPU.'

In [ ]:
# Clone ChartScope (or upload the backend/training/ folder manually)
!git clone --depth 1 https://github.com/YOUR_USERNAME/chartscope.git 2>/dev/null || true
%cd chartscope/backend

## 2. Install dependencies

Installs the backend stack plus training-only packages (`datasets`, `seqeval`, `evaluate`, `accelerate`).

In [ ]:
!pip install -q -r requirements.txt

## 3. Fine-tune PubMedBERT (3 epochs)

Trains a token-classification head on PubMedBERT with IOB labels (`O`, `B-Disease`, `I-Disease`). Metrics use **seqeval** entity-level strict P/R/F1.

In [ ]:
!python training/finetune_ner.py --epochs 3 --output_dir models/finetuned_ner

## 4. Baseline comparison

Evaluates the current ChartScope NER (`d4data/biomedical-ner-all`) on the **same NCBI-Disease test split** and appends results to `eval/finetune_metrics.json`.

In [ ]:
!python training/evaluate_baseline.py

## 5. Review metrics

In [ ]:
import json
from pathlib import Path

metrics_path = Path('eval/finetune_metrics.json')
metrics = json.loads(metrics_path.read_text())

print('=== Fine-tuned PubMedBERT (test) ===')
print(f"  F1:        {metrics['f1']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall:    {metrics['recall']:.4f}")

if 'baseline' in metrics:
    b = metrics['baseline']
    print('\n=== Baseline d4data/biomedical-ner-all (test) ===')
    print(f"  F1:        {b['f1']:.4f}")
    print(f"  Precision: {b['precision']:.4f}")
    print(f"  Recall:    {b['recall']:.4f}")
    delta = metrics['f1'] - b['f1']
    print(f"\n  Fine-tuned advantage: {delta:+.4f} F1")

## 6. (Optional) Push to Hugging Face Hub

Replace `YOUR_USERNAME/chartscope-ncbi-ner` with your repo id, then run the cell below.

In [ ]:
HUB_MODEL_ID = 'YOUR_USERNAME/chartscope-ncbi-ner'  # <-- change me

from huggingface_hub import login
login()  # paste your HF token when prompted

!python training/finetune_ner.py \
    --epochs 3 \
    --output_dir models/finetuned_ner \
    --push_to_hub \
    --hub_model_id {HUB_MODEL_ID}